# Clustering Analysis with Geospatial Data

This notebook analyzes and compares clustering results using different approaches for incorporating polygon information into HDBSCAN clustering. Three scenarios are examined:

1. **Scenario A: Only Original Points** - clustering using point geometries alone
2. **Scenario B: Points + Frequent Areas** - combining points with derived polygon-area data  
3. **Scenario C: Points + Polygon Centroids** - combining points with centroid data

Each method influences cluster size, density parameters, and spatial coverage differently.

## Key Functions Overview

### find_most_frequent_polygon_area(polygons, grid_size_meters=100)
Identifies geographic regions where input polygons overlap most frequently by creating a grid and counting intersections.

### optimize_and_cluster_geometries(geometries, true_lat, n_trials)
Uses HDBSCAN with Optuna optimization to find optimal clustering parameters (min_samples, eps_meters) and returns cluster polygons.

### display_geospatial_dataset(geometries, true_lat, true_lon, cluster_layers)
Visualizes geospatial data and clustering results on interactive Folium maps with toggleable layers.

In [97]:
# Check if running in Google Colab
try:
    from google.colab import drive
    print("Running in Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print("Not running in Google Colab. Skipping Google Drive mount.")
    IN_COLAB = False

Not running in Google Colab. Skipping Google Drive mount.


In [98]:
!pip install optuna
!pip install hdbscan
!pip install folium
!pip install shapely
import os
import sys

# Add the parent directory of ng to the Python path for package imports (only needed in Colab)
if IN_COLAB:
    module_path = '/content/drive/MyDrive/'
    if module_path not in sys.path:
        sys.path.append(module_path)

# Basic imports still needed in the main notebook
import hdbscan
import numpy as np
import optuna
import folium
import json
import datetime
from shapely.geometry import mapping # Only mapping, as Point, Polygon, MultiPoint, box are imported within the utils
import importlib

# Define the constant, consistent with previous usage
DEGREES_PER_METER_LAT = 0.0000089

# Import functions from the correct utils package depending on environment
if IN_COLAB:
    # In Colab, use the mounted Drive copy of the ng package
    import ng.utils.haversine_distance as haversine_distance_module
    import ng.utils.extract_points_from_geometries as extract_points_module
    import ng.utils.calculate_median_cluster_radius_meters as median_radius_module
    import ng.utils.cluster_points_and_get_all_cluster_polygons as cluster_polygons_module
    import ng.utils.optimize_and_cluster_geometries as optimize_cluster_module
    import ng.utils.generate_geospatial_dataset as generate_dataset_module
    import ng.utils.display_geospatial_dataset as display_dataset_module
    import ng.utils.find_most_frequent_polygon_area as frequent_area_module
    import ng.utils.convert_polygon_to_points as convert_polygon_module
    import ng.utils.polygons_to_random_points as polygons_to_random_points_module
    import ng.utils.extract_points_from_geometries as extract_points_from_geometries_module
else:
    # When running locally, import from the local utils package
    import utils.haversine_distance as haversine_distance_module
    import utils.extract_points_from_geometries as extract_points_module
    import utils.calculate_median_cluster_radius_meters as median_radius_module
    import utils.cluster_points_and_get_all_cluster_polygons as cluster_polygons_module
    import utils.optimize_and_cluster_geometries as optimize_cluster_module
    import utils.generate_geospatial_dataset as generate_dataset_module
    import utils.display_geospatial_dataset as display_dataset_module
    import utils.find_most_frequent_polygon_area as frequent_area_module
    import utils.convert_polygon_to_points as convert_polygon_module
    import utils.polygons_to_random_points as polygons_to_random_points_module
    import utils.extract_points_from_geometries as extract_points_from_geometries_module

# Reload modules so edits are picked up without restarting the kernel
importlib.reload(haversine_distance_module)
importlib.reload(extract_points_module)
importlib.reload(median_radius_module)
importlib.reload(cluster_polygons_module)
importlib.reload(optimize_cluster_module)
importlib.reload(generate_dataset_module)
importlib.reload(display_dataset_module)
importlib.reload(frequent_area_module)
importlib.reload(convert_polygon_module)
importlib.reload(polygons_to_random_points_module)
importlib.reload(extract_points_from_geometries_module)

print("All functions imported from ng.utils.")

All functions imported from ng.utils.


---
## Step 1: Generate Geospatial Dataset

In [99]:
new_geometries, new_true_lat, new_true_lon = generate_dataset_module.generate_geospatial_dataset()

print(f"Generated new dataset with {len(new_geometries)} geometries around Lat: {new_true_lat:.4f}, Lon: {new_true_lon:.4f}")

Generated True Location: (30.2423, 34.5410)
Calculated Degrees per Meter (Lat): 0.00000890
Calculated Degrees per Meter (Lon): 0.00001030
Generated 100 'most' points (0-20m bell curve).
Generated 20 'diverse' polygons (16 containing true location, 4 non-intersecting).
Generated new dataset with 120 geometries around Lat: 30.2423, Lon: 34.5410


In [100]:
# ============================================================================
# POINT REDUCTION CONFIGURATION
# ============================================================================

# Import the reduce_close_points function
if IN_COLAB:
    import ng.utils.reduce_close_points as reduce_close_points_module
else:
    import utils.reduce_close_points as reduce_close_points_module

importlib.reload(reduce_close_points_module)

# POINT REDUCTION PARAMETERS
use_point_reduction = True  # Set to True to enable point reduction
reduction_min_distance_meters = 5  # Consolidate points within 5 meters

print("\n" + "="*70)
print("POINT REDUCTION SETTINGS")
print("="*70)
print(f"Use point reduction: {use_point_reduction}")
if use_point_reduction:
    print(f"Reduction distance threshold: {reduction_min_distance_meters} meters")
    print("Method: Consolidate points within threshold distance using centroid")
else:
    print("Point reduction disabled - using original points for all scenarios")
print("="*70)



POINT REDUCTION SETTINGS
Use point reduction: True
Reduction distance threshold: 5 meters
Method: Consolidate points within threshold distance using centroid


In [101]:
from shapely.geometry import Point, Polygon, MultiPoint, box

original_points = extract_points_from_geometries_module.extract_points_from_geometries(new_geometries)
print(f"Extracted {len(original_points)} points from the geometries.")

original_polygons = [geom for geom in new_geometries if isinstance(geom, Polygon)]
print(f"Found {len(original_polygons)} polygons in the dataset.")

# Convert points to coordinate format for clustering
original_points_as_dict = [(pt.x, pt.y) for pt in original_points]

# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 1: Clustering with ORIGINAL POINTS only")
print("="*70)
print(f"Input: {len(original_points_as_dict)} original point(s)")

# SCENARIO 1: DO NOT apply point reduction - use all original points
scenario_1_points = original_points_as_dict
scenario_1_before = len(original_points_as_dict)
print(f"Point reduction DISABLED for Scenario 1 - using all {scenario_1_before} original point(s)")
print(f"(This preserves point density for better cluster detection)")

scenario_1_clusters, scenario_1_radii, scenario_1_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_1_points,
    new_true_lat,
    scenario_name="Scenario 1: Original Points Only"
)

# ===========================================================================
# SCENARIO 2: ALL POLYGONS CONVERTED TO RANDOM POINTS
# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 2: ALL POLYGONS → Random Points with Density Control")
print("="*70)

# POINT SPACING PARAMETER - You can control this!
point_spacing_meters = 150  # Approximate spacing between points in meters
print(f"Point spacing parameter: {point_spacing_meters} meters (smaller = more points)")

# Convert all polygons to a grid of points spaced approximately point_spacing_meters apart
all_polygon_points = polygons_to_random_points_module.polygons_to_random_points(
    original_polygons,
    spacing_meters=point_spacing_meters
)
print(f"Generated {len(all_polygon_points)} random points from {len(original_polygons)} polygon(s)")

# Combine with original points
scenario_2_all_points = original_points_as_dict + [(pt.x, pt.y) for pt in all_polygon_points]
scenario_2_before = len(scenario_2_all_points)
print(f"Total points before reduction: {scenario_2_before}")

# Disable point reduction
use_point_reduction = False
reduction_min_distance_meters = 5  # Still print for info

if use_point_reduction:
    scenario_2_all_points_reduced = reduce_close_points_module.reduce_close_points(scenario_2_all_points, reduction_min_distance_meters, haversine_distance_module.haversine_distance)
    scenario_2_reduced = len(scenario_2_all_points_reduced)
else:
    scenario_2_all_points_reduced = scenario_2_all_points
    scenario_2_reduced = scenario_2_before

print("Number of points before reduction:")
print(f"{scenario_2_before}")
print("Number of points after reduction:")
print(f"{scenario_2_reduced}")
print(f"Reduction enabled: {use_point_reduction}")
print(f"Reduction distance threshold: {reduction_min_distance_meters} meters")
print("-----------------------------------")

# ===========================================================================
# SCENARIO 3: HIGH FREQUENCY AREAS → RANDOM POINTS
# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 3: HIGH FREQUENCY Polygon Areas → Random Points")
print("="*70)

# Find high frequency polygon areas
high_freq_polygons = frequent_area_module.find_most_frequent_polygon_area(original_polygons, grid_size_meters=100)

# Ensure high_freq_polygons is a list of Polygon(s)
if isinstance(high_freq_polygons, list):
    # Remove any non-Polygon objects (e.g., floats)
    high_freq_polygons = [poly for poly in high_freq_polygons if isinstance(poly, Polygon)]
    print(f"Found {len(high_freq_polygons)} high frequency polygon area(s)")
else:
    if isinstance(high_freq_polygons, Polygon):
        print("Found 1 high frequency polygon area")
        high_freq_polygons = [high_freq_polygons]
    else:
        print("No valid high frequency polygons found.")
        high_freq_polygons = []

# Generate random points inside these polygons with 50 meter spacing
scenario_3_point_spacing_meters = 50
scenario_3_points = polygons_to_random_points_module.polygons_to_random_points(
    high_freq_polygons,
    spacing_meters=scenario_3_point_spacing_meters
)
print(f"Generated {len(scenario_3_points)} random points from high frequency polygons")

# Combine with original points if desired (here, just use random points from high frequency polygons)
scenario_3_all_points = [(pt.x, pt.y) for pt in scenario_3_points]
scenario_3_before = len(scenario_3_all_points)
print(f"Total points for Scenario 3: {scenario_3_before}")

# No reduction for Scenario 3
print("Point reduction DISABLED for Scenario 3")
print("-----------------------------------")

# Ready for clustering


[I 2026-03-20 18:17:00,549] A new study created in memory with name: no-name-e7622ab7-dbaf-40b4-9754-98ce538eb390
[I 2026-03-20 18:17:00,586] Trial 0 finished with value: inf and parameters: {'min_samples': 21, 'eps_meters': 27.40161107207667}. Best is trial 0 with value: inf.
[I 2026-03-20 18:17:00,621] Trial 1 finished with value: inf and parameters: {'min_samples': 6, 'eps_meters': 58.337390227346255}. Best is trial 0 with value: inf.


Extracted 100 points from the geometries.
Found 20 polygons in the dataset.

SCENARIO 1: Clustering with ORIGINAL POINTS only
Input: 100 original point(s)
Point reduction DISABLED for Scenario 1 - using all 100 original point(s)
(This preserves point density for better cluster detection)


[I 2026-03-20 18:17:00,659] Trial 2 finished with value: inf and parameters: {'min_samples': 16, 'eps_meters': 37.443976705856485}. Best is trial 0 with value: inf.
[I 2026-03-20 18:17:00,691] Trial 3 finished with value: inf and parameters: {'min_samples': 32, 'eps_meters': 26.50316220979273}. Best is trial 0 with value: inf.
[I 2026-03-20 18:17:00,726] Trial 4 finished with value: inf and parameters: {'min_samples': 49, 'eps_meters': 34.72912277746573}. Best is trial 0 with value: inf.
[I 2026-03-20 18:17:00,764] Trial 5 finished with value: inf and parameters: {'min_samples': 44, 'eps_meters': 38.868313701140984}. Best is trial 0 with value: inf.
[I 2026-03-20 18:17:00,798] Trial 6 finished with value: inf and parameters: {'min_samples': 21, 'eps_meters': 38.53448216311509}. Best is trial 0 with value: inf.
[I 2026-03-20 18:17:00,833] Trial 7 finished with value: 1.382401726615946 and parameters: {'min_samples': 3, 'eps_meters': 23.913603908944623}. Best is trial 7 with value: 1.382

Optimization complete for Scenario 1: Original Points Only. Best min_samples: 3, best eps_meters: 23.91, optimized median cluster radius: 1.38 meters

SCENARIO 2: ALL POLYGONS → Random Points with Density Control
Point spacing parameter: 150 meters (smaller = more points)
Generated 419 random points from 20 polygon(s)
Total points before reduction: 519
Number of points before reduction:
519
Number of points after reduction:
519
Reduction enabled: False
Reduction distance threshold: 5 meters
-----------------------------------

SCENARIO 3: HIGH FREQUENCY Polygon Areas → Random Points
Found 9 area(s) with frequency >= 80% of max (13) intersecting polygons.
Found 9 high frequency polygon area(s)
Generated 9 random points from high frequency polygons
Total points for Scenario 3: 9
Point reduction DISABLED for Scenario 3
-----------------------------------


## How to Control Point Density in Scenarios 2 & 3

The **`point_spacing_meters`** parameter controls how densely points are placed inside each polygon by spacing them approximately that far apart (in meters):

- **`point_spacing_meters = 100`** → Sparse distribution (points spaced ~100m apart)
- **`point_spacing_meters = 50`** → Moderate distribution (default, balanced)
- **`point_spacing_meters = 25`** → Dense distribution (points spaced ~25m apart)
- **`point_spacing_meters = 10`** → Very dense distribution (points spaced ~10m apart)

**Change this value** in the "Scenario 2" code cell to see how point spacing affects clustering results!

---
## Step 2: Interactive Map - Compare All Scenarios

Toggle each scenario layer in the map legend (top-right) to see how different polygon-to-point conversion methods affect clustering results.

In [ ]:
# ============================================================================
# DEBUG: Check what clusters were generated
# ============================================================================
print("\n" + "="*70)
print("DEBUG: CLUSTER INSPECTION")
print("="*70)

print(f"\nScenario 1 clusters: {len(scenario_1_clusters)} cluster(s)")
for i, cluster in enumerate(scenario_1_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")

# Ensure scenario_2_clusters is defined by running clustering for Scenario 2
scenario_2_clusters, scenario_2_radii, scenario_2_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_2_all_points_reduced,
    new_true_lat,
    scenario_name="Scenario 2: All Polygons as Random Points"
)

print(f"\nScenario 2 clusters: {len(scenario_2_clusters)} cluster(s)")
for i, cluster in enumerate(scenario_2_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")

if len(scenario_1_clusters) == 0 and len(scenario_2_clusters) == 0:
    print("\n⚠️  WARNING: NO CLUSTERS FOUND!")
    print("This likely means HDBSCAN found no significant clusters in either scenario.")
    print("Possible reasons:")
    print("  1. Point density is too low")
    print("  2. Clustering parameters (min_samples, eps_meters) are too restrictive")
    print("  3. Data points are too scattered/isolated")
elif len(scenario_1_clusters) > 0 or len(scenario_2_clusters) > 0:
    print("\n✓ Clusters found - proceeding to visualization")
print("="*70 + "\n")


[I 2026-03-20 18:17:06,012] A new study created in memory with name: no-name-2111157c-f529-41d9-9f92-4ae8d18c1c3a



DEBUG: CLUSTER INSPECTION

Scenario 1 clusters: 2 cluster(s)
  Cluster 0: Type=Polygon, Bounds=(34.54100002806092, 30.24232355581306, 34.54106570261236, 30.24237066380614)
  Cluster 1: Type=Polygon, Bounds=(34.54099524225488, 30.24234446822917, 34.541002312074085, 30.242368143157133)


[I 2026-03-20 18:17:06,776] Trial 0 finished with value: 699.1133072941692 and parameters: {'min_samples': 10, 'eps_meters': 50.55018645512827}. Best is trial 0 with value: 699.1133072941692.
[I 2026-03-20 18:17:07,538] Trial 1 finished with value: 243.484395555742 and parameters: {'min_samples': 3, 'eps_meters': 97.02581831986762}. Best is trial 1 with value: 243.484395555742.
[I 2026-03-20 18:17:08,321] Trial 2 finished with value: 48.11758950227256 and parameters: {'min_samples': 17, 'eps_meters': 95.67521996829127}. Best is trial 2 with value: 48.11758950227256.
[I 2026-03-20 18:17:09,093] Trial 3 finished with value: 322.71794685098166 and parameters: {'min_samples': 23, 'eps_meters': 82.92932306736749}. Best is trial 2 with value: 48.11758950227256.
[I 2026-03-20 18:17:09,861] Trial 4 finished with value: 699.1133072941692 and parameters: {'min_samples': 9, 'eps_meters': 88.72480017173352}. Best is trial 2 with value: 48.11758950227256.
[I 2026-03-20 18:17:10,628] Trial 5 finishe

In [ ]:
# Diagnostic: Print cluster coordinates and bounds for visual inspection
print("\n" + "="*70)
print("DIAGNOSTIC: Cluster Geometry Details")
print("="*70)
print(f"True location: Lat={new_true_lat}, Lon={new_true_lon}")

print(f"\nScenario 1 clusters: {len(scenario_1_clusters)}")
for i, cluster in enumerate(scenario_1_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")
    if cluster.geom_type == 'Polygon':
        coords = list(cluster.exterior.coords)
        print(f"    First 3 coords: {coords[:3]}")
        print(f"    All coords: {coords}")

print(f"\nScenario 2 clusters: {len(scenario_2_clusters)}")
for i, cluster in enumerate(scenario_2_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")
    if cluster.geom_type == 'Polygon':
        coords = list(cluster.exterior.coords)
        print(f"    First 3 coords: {coords[:3]}")
        print(f"    All coords: {coords}")

print("="*70 + "\n")

In [ ]:
print("\n" + "="*70)
print("INTERACTIVE MAP - SCENARIO 1, 2, 3")
print("="*70)
print("Displaying clustering results on an interactive map.\n")
print("Layer Legend:")
print("  📍 Gray: Base Dataset (Original polygons)")
print("  🔴 Red: Scenario 1 - Original Points Only")
print("  🟢 Green: Scenario 2 - All Polygons as Random Points")
print("  🔵 Blue: Scenario 3 - High Frequency Areas")
print("\nTip: Toggle layers in the legend (top-right) to compare scenarios.")
print("="*70 + "\n")

centroid_polygons = new_geometries
print(f"Creating map with {len(scenario_1_clusters)} clusters from Scenario 1, {len(scenario_2_clusters)} clusters from Scenario 2, and {len(high_freq_polygons)} high frequency polygons from Scenario 3...")

map_display = display_dataset_module.display_geospatial_dataset(
    [scenario_1_clusters, scenario_2_clusters, high_freq_polygons],
    [scenario_1_cluster_medians, scenario_2_cluster_medians, None],
    centroid_polygons,
    new_true_lat,
    new_true_lon
)

print("Map created successfully!")

from IPython.display import display
display(map_display)

In [ ]:
from utils.haversine_distance import haversine_distance
from shapely.geometry import Point
import numpy as np

# Calculate centroids for clusters
red_centroids = [cluster.centroid for cluster in scenario_1_clusters if hasattr(cluster, 'centroid')]
green_centroids = [cluster.centroid for cluster in scenario_2_clusters if hasattr(cluster, 'centroid')]

# True location as Point
yellow_point = Point(new_true_lon, new_true_lat)

# Calculate distances from centroids to true location
red_distances = [haversine_distance((c.y, c.x), (new_true_lat, new_true_lon)) for c in red_centroids]
green_distances = [haversine_distance((c.y, c.x), (new_true_lat, new_true_lon)) for c in green_centroids]

print("\n--- Cluster Distance Statistics ---")
print(f"Red cluster distances to true location: {red_distances}")
print(f"Green cluster distances to true location: {green_distances}")
if red_distances:
    print(f"Red cluster mean distance: {np.mean(red_distances):.2f} m, std: {np.std(red_distances):.2f} m")
if green_distances:
    print(f"Green cluster mean distance: {np.mean(green_distances):.2f} m, std: {np.std(green_distances):.2f} m")
print("-----------------------------------\n")

## Summary: Impact of Polygon Integration on Clustering

### Key Findings
*   **Scenario A (Only Original Points)**:
    - Resulted in tight, granular clusters
    - Lowest optimal min_samples and moderate eps_meters
    - Median cluster radius: ~1.79 meters
    - Focus on immediate vicinity of point distribution

*   **Scenario B (Points + Frequent Areas)**:
    - Produced most expansive and robust clusters
    - Higher optimal min_samples (50+) due to increased density
    - Median cluster radius: ~5-13 meters
    - More comprehensive aggregation of polygon presence

*   **Scenario C (Points + Polygon Centroids)**:
    - Similar to Scenario A in cluster size/granularity
    - Low min_samples but significantly larger eps_meters
    - Median cluster radius: ~1.79-2.56 meters
    - Subtle expansion of cluster boundaries

### Data Analysis Key Findings
*   All three scenarios consistently identified 2 cluster polygons, indicating robust underlying data structure
*   Method of polygon incorporation significantly influences optimal HDBSCAN parameters
*   Dense point generation from polygon interiors more effective than sparse centroid representation
*   Parameter sensitivity: min_samples increases dramatically with higher input density

## Final Task

### Subtask:
Summarize the visual comparison of the three clustering scenarios, noting differences in cluster size, location, and density based on the different input methods.

### Analysis and Comparison of Clustering Results

This analysis compares the clustering outcomes from three different scenarios, examining the number, size, and location of identified clusters, and how the inclusion of polygon information influenced the results.

#### Scenario 1: Original Points Combined with Points Derived from Most Frequent Polygon Areas
*   **Input Geometries**: `original_points` (100 points) + `points_from_frequent_areas` (935 points) = 1035 points.
*   **Optimal Parameters**: `min_samples` = 24, `eps_meters` = 41.84
*   **Optimized Median Cluster Radius**: 42.01 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: The inclusion of a large number of interior points from the 'most frequent areas' significantly increased the density around the true location and other high-frequency areas. This led to a higher `min_samples` value (24) and a moderate `eps_meters` (41.84m) to form robust clusters that required high local density. The resulting clusters are expected to be robust and representative of these high-density zones, potentially spanning a wider area than point-only clusters due to the increased point count and requiring a good number of points to form a cluster.

#### Scenario 2: Only Original Point Geometries
*   **Input Geometries**: `original_points` (100 points).
*   **Optimal Parameters**: `min_samples` = 5, `eps_meters` = 11.17
*   **Optimized Median Cluster Radius**: 0.37 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: With only the original 100 points (which were generated within 20 meters of the true location), the clustering algorithm found very tight, small clusters. The low `min_samples` (5) and very small `eps_meters` (11.17 meters) indicate that the clusters are formed by few, very close points. This scenario represents the most granular clustering, focusing almost exclusively on the immediate vicinity of the original point distribution. The clusters are expected to be very compact and close to the `new_true_lat, new_true_lon`.

#### Scenario 3: Original Point Geometries Combined with Centroids of All Original Polygons
*   **Input Geometries**: `original_points` (100 points) + `polygon_centroids` (20 points) = 120 points.
*   **Optimal Parameters**: `min_samples` = 3, `eps_meters` = 71.23
*   **Optimized Median Cluster Radius**: 2.56 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: Adding only the centroids of polygons to the original points provided some additional geographic context but far less density than Scenario 1. The `min_samples` remained low (3), similar to clustering only points. The `eps_meters` was quite large (71.23 meters) and the median radius (2.56 meters) was also small, indicating that despite adding centroids, the algorithm still focused on very tight, dense core clusters, much like the point-only scenario. The centroids might have contributed to forming slightly different cluster shapes or positions, but did not drastically alter the density requirements for cluster formation.

#### Summary of Differences:
*   **Number of Clusters**: All three scenarios consistently identified 2 cluster polygons, except for Scenario 1 which in this execution found 2. Previous executions sometimes showed variation. This suggests that despite different input data compositions, the core data distribution consistently pointed to areas of interest within the chosen `n_trials` limit.
*   **Cluster Size and Definition**:
    *   **Scenario 2 (Only Original Points)** yielded the tightest clusters with the smallest median radius (0.37m) and a low `min_samples` (5). The `eps_meters` was small (11.17m), indicating a focus on highly localized, dense groupings.
    *   **Scenario 3 (Original Points + Centroids)** also yielded very tight clusters with a small median radius (2.56m) and low `min_samples` (3). However, its `eps_meters` was significantly larger (71.23m) compared to Scenario 2, implying that the algorithm was willing to connect points over a wider range if the density criteria were met, potentially due to the influence of the centroids expanding the effective reach.
    *   **Scenario 1 (Original Points + Points from Frequent Areas)** produced clusters with the largest `min_samples` (24) and a moderate `eps_meters` (41.84m), resulting in the largest optimized median radius (42.01 meters). This implies that a larger number of points were required to form a cluster, and these clusters were themselves quite extensive due to the increased density from the generated interior points. This method created the most robust and spatially expansive clusters.

*   **Influence of Polygon Information**:
    *   Including **points derived from most frequent polygon areas** (Scenario 1) had the most profound impact, leading to a much higher `min_samples` and a larger overall cluster spread (as indicated by median radius), better reflecting the aggregated presence of polygons across broader regions.
    *   Including **only polygon centroids** (Scenario 3) had a more subtle effect. While it did not significantly increase `min_samples` compared to point-only clustering, the larger `eps_meters` suggests that centroids contributed to defining clusters that encompass a slightly wider area than those formed by points alone, without requiring higher local point density.
